#cell 1

In [ ]:
!pip install -q "numpy<2.0.0" insightface==0.2.1 onnxruntime-gpu
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python Pillow tqdm
print("✅ Installations completed. 🚨 CRITICAL: Go to Runtime -> Restart session NOW!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 25.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompa

#cell2


In [ ]:
# תא 2: הכנת הנתונים והמשקלים (Master Setup)
import os, shutil, glob, json, random
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
%cd /content
!rm -rf /content/SimSwap
!git clone -q https://github.com/neuralchen/SimSwap
%cd /content/SimSwap

print("Downloading models and weights...")
!mkdir -p insightface_func/models/antelope
!wget -q -O antelope.zip https://huggingface.co/haikumonster/antelope/resolve/main/antelope.zip
!unzip -n -q antelope.zip -d insightface_func/models/antelope/
!rm antelope.zip

!wget -q -O checkpoints.zip https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
!unzip -n -q checkpoints.zip -d ./
!rm checkpoints.zip
!mkdir -p checkpoints
!mv people checkpoints/

!mkdir -p arcface_model
!wget -q -O arcface_model/arcface_checkpoint.tar https://huggingface.co/netrunner-exe/SimSwap-models/resolve/main/arcface_checkpoint.tar

!mkdir -p parsing_model/checkpoint
!wget -q -O parsing_model/checkpoint/79999_iter.pth https://huggingface.co/vivym/face-parsing-bisenet/resolve/main/79999_iter.pth

print("Fixing InsightFace nested directories...")
antelope_dir = '/content/SimSwap/insightface_func/models/antelope'
for f in glob.glob(f'{antelope_dir}/*/*.onnx'):
    shutil.move(f, os.path.join(antelope_dir, os.path.basename(f)))

print("Extracting images to local SSD...")
archive_path = '/content/drive/MyDrive/Deepfake_Project/archive.zip'
local_real_dir = '/content/dataset/real_extracted'
os.makedirs(local_real_dir, exist_ok=True)
!unzip -n -q "{archive_path}" -d "{local_real_dir}"

# טעינת רשימת התמונות התקינות מ-Drive
print("Loading pre-scanned valid images from Drive...")
!cp /content/drive/MyDrive/Deepfake_Project/valid_images.json /content/SimSwap/valid_images.json

with open('/content/SimSwap/valid_images.json') as f:
    images = json.load(f)

print(f"Loaded {len(images)} valid images")

random.seed(42)
images = random.sample(images, min(25000, len(images)))
pairs = [{"source": images[i], "target": images[(i+1) % len(images)]} for i in range(len(images))]

with open('/content/SimSwap/pairs.json', 'w') as f:
    json.dump(pairs, f, indent=4)

print(f"Generated {len(pairs)} pairs from valid images only. Ready for generation.")

Mounted at /content/drive
/content
/content/SimSwap
Fixing InsightFace nested directories...
Extracting images to local SSD...
Loading pre-scanned valid images from Drive...
Loaded 25195 valid images
Generated 25000 pairs from valid images only. Ready for generation.


#2.5

In [ ]:
# תא 2.5: סינון תמונות עם פנים + בדיקת הניצולים
import cv2, glob, json, os
from tqdm import tqdm
from insightface_func.face_detect_crop_single import Face_detect_crop

# --- שלב 1: סריקה ראשונית ---
app = Face_detect_crop(name='antelope', root='./insightface_func/models')
app.prepare(ctx_id=0, det_thresh=0.25, det_size=(640,640))

local_real_dir = '/content/dataset/real_extracted'
all_images = list(set(
    glob.glob(f'{local_real_dir}/**/*.jpg', recursive=True) +
    glob.glob(f'{local_real_dir}/**/*.png', recursive=True)
))

print(f"Total images: {len(all_images)}")

valid_images = []
invalid_images = []

for img_path in tqdm(all_images, desc="Scanning for faces"):
    img = cv2.imread(img_path)
    if img is None:
        continue
    res = app.get(img, 224)
    if res is not None and len(res[0]) > 0:
        valid_images.append(img_path)
    else:
        invalid_images.append(img_path)

print(f"\nValid (has face): {len(valid_images)}")
print(f"Invalid (no face): {len(invalid_images)}")
print(f"Success rate: {len(valid_images)/len(all_images)*100:.1f}%")

# --- שלב 2: בדיקת ניצול על 100 תמונות שנפסלו ---
print("\nTesting rescue with aggressive params on 100 invalid images...")
app2 = Face_detect_crop(name='antelope', root='./insightface_func/models')
app2.prepare(ctx_id=0, det_thresh=0.2, det_size=(1280,1280))

rescued = 0
rescued_paths = []
for img_path in tqdm(invalid_images[:100], desc="Rescue attempt"):
    img = cv2.imread(img_path)
    if img is None:
        continue
    res = app2.get(img, 224)
    if res is not None and len(res[0]) > 0:
        rescued += 1
        rescued_paths.append(img_path)

print(f"\nRescued: {rescued}/100 ({rescued}%)")
if rescued >= 30:
    print("Worth rescanning all invalid images with aggressive params!")
else:
    print("Not worth it - invalid images are genuinely faceless. Proceeding with valid only.")

# --- שלב 3: שמירה ---
with open('/content/SimSwap/valid_images.json', 'w') as f:
    json.dump(valid_images, f)
with open('/content/SimSwap/invalid_images.json', 'w') as f:
    json.dump(invalid_images, f)

drive_dir = '/content/drive/MyDrive/Deepfake_Project'
if os.path.exists(drive_dir):
    !cp /content/SimSwap/valid_images.json "{drive_dir}/valid_images.json"
    !cp /content/SimSwap/invalid_images.json "{drive_dir}/invalid_images.json"
    print("Saved to Drive!")
else:
    print("Drive not mounted - saved locally only")

input mean and std: 127.5 127.5
find model: ./insightface_func/models/antelope/glintr100.onnx recognition
find model: ./insightface_func/models/antelope/scrfd_10g_bnkps.onnx detection
set det-size: (640, 640)
Total images: 30000


Scanning for faces:  41%|████▏     | 12392/30000 [1:10:04<1:24:18,  3.48it/s]

#cell 3

In [ ]:
# תא 3: The Ultimate Batch Swapper
%%writefile /content/SimSwap/batch_swapper.py
import sys

# Mock command line arguments
sys.argv = [
    'test',
    '--name', 'people',
    '--Arc_path', 'arcface_model/arcface_checkpoint.tar',
    '--pic_a_path', 'dummy',
    '--video_path', 'dummy',
    '--output_path', '/content/dataset/fake_simswap',
    '--crop_size', '224',
    '--no_simswaplogo',
]

import os, cv2, json, warnings, subprocess
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn.functional as F
from torchvision import transforms
from options.test_options import TestOptions
from models.models import create_model
from insightface_func.face_detect_crop_single import Face_detect_crop
from util.reverse2original import reverse2wholeimage

warnings.filterwarnings("ignore")

def main():
    opt = TestOptions().parse()
    opt.isTrain = False

    out_dir = '/content/dataset/fake_simswap'
    os.makedirs(out_dir, exist_ok=True)

    with open('/content/SimSwap/pairs.json', 'r') as f:
        pairs = json.load(f)

    torch.nn.Module.dump_patches = True
    model = create_model(opt)
    model.eval()

    app = Face_detect_crop(name='antelope', root='./insightface_func/models')
    app.prepare(ctx_id=0, det_thresh=0.3, det_size=(640, 640))

    trans = transforms.ToTensor()

    error_count = 0
    success_count = 0

    with torch.no_grad():
        for i, pair in enumerate(tqdm(pairs, desc="Swapping Faces")):
            basename = os.path.splitext(os.path.basename(pair['target']))[0] + '.jpg'
            out_path = os.path.join(out_dir, basename)

            if os.path.exists(out_path):
                success_count += 1
                continue

            try:
                img_a = cv2.imread(pair['source'])
                img_b = cv2.imread(pair['target'])
                if img_a is None or img_b is None:
                    raise ValueError("Failed to read image")

                res_a = app.get(img_a, opt.crop_size)
                if res_a is None: raise ValueError("No face in source")
                img_a_crop, _ = res_a

                res_b = app.get(img_b, opt.crop_size)
                if res_b is None: raise ValueError("No face in target")
                img_b_crop_list, b_mat_list = res_b

                img_a_t = trans(Image.fromarray(cv2.cvtColor(img_a_crop[0], cv2.COLOR_BGR2RGB)))\
                    .view(-1, 3, opt.crop_size, opt.crop_size).cuda()
                img_a_112 = F.interpolate(img_a_t, size=(112,112), mode='bilinear', align_corners=False)
                latent_id = F.normalize(model.netArc(img_a_112), p=2, dim=1)

                img_b_t = trans(Image.fromarray(cv2.cvtColor(img_b_crop_list[0], cv2.COLOR_BGR2RGB)))\
                    .view(-1, 3, opt.crop_size, opt.crop_size).cuda()
                img_b_112 = F.interpolate(img_b_t, size=(112,112), mode='bilinear', align_corners=False)
                latent_att = F.normalize(model.netArc(img_b_112), p=2, dim=1)

                img_fake = model(img_b_t, img_b_t, latent_id, latent_att)

                final_img = reverse2wholeimage(
                    img_b_crop_list, img_fake, b_mat_list,
                    opt.crop_size, img_b,
                    logoclass=None,
                    save_path=out_path,
                    no_simswaplogo=True
                )

                if final_img is None:
                    raise ValueError("reverse2wholeimage returned None")

                success_count += 1

                if success_count > 0 and success_count % 1000 == 0:
                    if os.path.exists("/content/drive/MyDrive/Deepfake_Project"):
                        subprocess.Popen([
                            "zip", "-r", "-q", "-j",
                            f"/content/drive/MyDrive/Deepfake_Project/SimSwap_Backup_{success_count}.zip",
                            out_dir
                        ])

            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    tqdm.write(f"\nError {i}: {e}")
                continue

            if i == 10 and success_count == 0:
                raise RuntimeError(f"FAILSAFE: 0 successes in first 10. Last error visible above.")

    print(f"\nDone! Success: {success_count}, Errors: {error_count}")

if __name__ == '__main__':
    main()

Writing /content/SimSwap/batch_swapper.py


#cell 3.5

In [ ]:
# תא 3.5: פאצ'ים לקוד המקור של SimSwap
print("Patching SimSwap source code...")
import os, re

# 1. תיקוני תאימות NumPy
!sed -i 's/np\.float\b/float/g' /content/SimSwap/util/reverse2original.py
!sed -i 's/np\.int\b/int/g' /content/SimSwap/util/reverse2original.py
!sed -i 's/np\.complex\b/complex/g' /content/SimSwap/util/reverse2original.py
!sed -i 's/np\.float\b/float/g' /content/SimSwap/models/fs_model.py
!sed -i 's/np\.float\b/float/g' /content/SimSwap/insightface_func/face_detect_crop_single.py

r2o_path = '/content/SimSwap/util/reverse2original.py'
with open(r2o_path, 'r') as f:
    content = f.read()

# 2. תיקון באג הלוגו - מונע כפילות
content = re.sub(
    r'[ \t]*(if logoclass is not None: )*final_img = logoclass\.apply_frames\(final_img\)',
    '        if logoclass is not None: final_img = logoclass.apply_frames(final_img)',
    content
)

# 3. תיקון return חסר
old = '    cv2.imwrite(save_path, final_img)\n'
new = '    if save_path:\n        cv2.imwrite(save_path, final_img)\n    return final_img\n'
if old in content and 'return final_img' not in content:
    content = content.replace(old, new)
    print("Fixed: added return final_img")
else:
    print("return already exists or pattern not found")

with open(r2o_path, 'w') as f:
    f.write(content)

# וידוא
result = [l for l in content.splitlines() if 'logoclass' in l and 'apply_frames' in l]
print(f"Logoclass lines: {result}")
return_lines = [l for l in content.splitlines() if 'return final_img' in l]
print(f"Return lines: {return_lines}")

# 4. תיקון PyTorch 2.6+
fs_model_path = '/content/SimSwap/models/fs_model.py'
with open(fs_model_path, 'r') as f:
    patch = f.read()

old_load = 'netArc_checkpoint = torch.load(netArc_checkpoint, map_location=torch.device("cpu"))'
new_load = 'netArc_checkpoint = torch.load(netArc_checkpoint, map_location=torch.device("cpu"), weights_only=False)'

if old_load in patch:
    patch = patch.replace(old_load, new_load)
    with open(fs_model_path, 'w') as f:
        f.write(patch)
    print("Fixed torch.load in fs_model.py")
else:
    print("torch.load already patched")

print("All patches applied. Ready for Cell 4.")

Patching SimSwap source code...
Fixed: added return final_img
Logoclass lines: ['        if logoclass is not None: final_img = logoclass.apply_frames(final_img)']
Return lines: ['    return final_img']
Fixed torch.load in fs_model.py
All patches applied. Ready for Cell 4.


#Cell 4

In [ ]:
# Cell 4: Execute the Pipeline
%cd /content/SimSwap
!python batch_swapper.py

/content/SimSwap
------------ Options -------------
Arc_path: arcface_model/arcface_checkpoint.tar
aspect_ratio: 1.0
batchSize: 8
checkpoints_dir: ./checkpoints
cluster_path: features_clustered_010.npy
crop_size: 224
data_type: 32
dataroot: ./datasets/cityscapes/
display_winsize: 512
engine: None
export_onnx: None
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
how_many: 50
id_thres: 0.03
image_size: 224
input_nc: 3
instance_feat: False
isTrain: False
label_feat: False
label_nc: 0
latent_size: 512
loadSize: 1024
load_features: False
local_rank: 0
max_dataset_size: inf
multisepcific_dir: ./demo_file/multispecific
nThreads: 2
n_blocks_global: 6
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 3
n_local_enhancers: 1
name: people
nef: 16
netG: global
ngf: 64
niter_fix_global: 0
no_flip: False
no_instance: False
no_simswaplogo: True
norm: batch
norm_G: spectralspadesyncbatch3x3
ntest: inf
onnx: None
output_nc: 3
output_path: /content/dataset/fake_simswap
phase: tes

#cell 5

In [ ]:
# תא 5: אריזה, ספירה, וגיבוי בטוח במיוחד לדרייב
import os
import shutil

output_dir = '/content/dataset/fake_simswap/'
zip_path = '/content/CelebA_HQ_SimSwap_Final.zip'
drive_path = '/content/drive/MyDrive/Deepfake_Project/CelebA_HQ_SimSwap_Final.zip'

print("🔍 Running Final Sanity Checks...")
# 1. ספירת קבצים לוודא שהכל נוצר
if os.path.exists(output_dir):
    num_files = len(os.listdir(output_dir))
    print(f"📊 Found {num_files} generated images in local storage.")
else:
    raise FileNotFoundError("🚨 Output directory not found! Did generation fail?")

# 2. דחיסה בטוחה (ללא RAM)
print("📦 Packing files directly via Bash to prevent RAM overload...")
!zip -r -q -j {zip_path} {output_dir}

# בדיקת גודל הקובץ המקומי
if os.path.exists(zip_path):
    local_size = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✅ Local ZIP created successfully. Size: {local_size:.2f} MB")
else:
    raise FileNotFoundError("🚨 ZIP file creation failed!")

# 3. העתקה בטוחה לדרייב (חובה cp ולא mv!)
print("🚚 Copying to Google Drive (Using 'cp' to ensure local backup stays safe)...")
!cp {zip_path} {drive_path}

# 4. וידוא הגעה והשוואת גדלים כדי להיות בטוחים ב-100%
if os.path.exists(drive_path):
    drive_size = os.path.getsize(drive_path) / (1024 * 1024)
    print(f"✅ ZIP detected on Drive. Size: {drive_size:.2f} MB")

    # התיקון שלך: סטייה מותרת של עד 10 מגה-בייט למניעת התראות שווא על קבצי ענק
    if abs(local_size - drive_size) < 10.0:
        print("\n🎉 MISSION ACCOMPLISHED! PIPELINE FULLY COMPLETED AND SECURED! 🏆")
        print("You can now safely download the ZIP from your Google Drive website to your local PC.")
    else:
        print("\n⚠️ WARNING: The file size on Drive differs from the local file significantly.")
        print("Do NOT close Colab! Download it manually from the left 'Files' panel.")
else:
    print("\n🚨 Failed to copy to Drive! The ZIP is still safe on Colab local storage.")
    print("Download it manually from the left 'Files' panel before closing the runtime.")

#cell 6

In [ ]:
# תא 1: יצירת מאגר מאוזן (Real vs Fake) לקראת בחינת ה-Zero-Shot
import os
import json
import random
import shutil
from tqdm.auto import tqdm
from google.colab import drive

print("🔌 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- הגדרת נתיבים ---
drive_dir = '/content/drive/MyDrive/Deepfake_Project'
fake_zip_path = f'{drive_dir}/CelebA_HQ_SimSwap_Final.zip'
real_zip_path = f'{drive_dir}/archive.zip'
valid_json_path = f'{drive_dir}/valid_images.json'

# תיקיות עבודה זמניות וסופיות
extract_real_temp = '/content/temp_real_extracted'
extract_fake_temp = '/content/temp_fake_extracted'
final_dataset_dir = '/content/Balanced_Dataset'
final_real_dir = f'{final_dataset_dir}/real'
final_fake_dir = f'{final_dataset_dir}/fake'
final_zip_drive = f'{drive_dir}/CelebA_HQ_Balanced_ZeroShot.zip'

# ניקוי תיקיות קודמות (אם נשאר משהו)
!rm -rf {extract_real_temp} {extract_fake_temp} {final_dataset_dir}
os.makedirs(final_real_dir, exist_ok=True)
os.makedirs(final_fake_dir, exist_ok=True)

# --- שלב 1: טיפול בתמונות המזויפות ---
print("\n📦 Extracting Fake images...")
!unzip -q {fake_zip_path} -d {extract_fake_temp}

# מציאת כל התמונות המזויפות שחולצו
fake_images = []
for root, dirs, files in os.walk(extract_fake_temp):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            fake_images.append(os.path.join(root, file))

num_fakes = len(fake_images)
print(f"📊 Found {num_fakes} FAKE images.")

if num_fakes == 0:
    raise ValueError("🚨 No fake images found! Check your fake zip file.")

# העברת המזויפות לתיקיית היעד הסופית ('fake')
print("🚚 Moving Fake images to final directory...")
for img in tqdm(fake_images, desc="Copying Fakes"):
    shutil.move(img, os.path.join(final_fake_dir, os.path.basename(img)))

# --- שלב 2: חילוץ התמונות האמיתיות ---
print("\n📦 Extracting Real images...")
# אנחנו מחלצים בדיוק לאותו נתיב שהיה ב-JSON המקורי כדי שהנתיבים יתאימו
local_original_real_dir = '/content/dataset/real_extracted'
!rm -rf {local_original_real_dir}
os.makedirs(local_original_real_dir, exist_ok=True)
!unzip -n -q {real_zip_path} -d {local_original_real_dir}

# --- שלב 3: טעינת ה-JSON ודגימה מאוזנת ---
print("\n📋 Loading valid JSON and balancing data...")
with open(valid_json_path, 'r') as f:
    valid_real_paths = json.load(f)

print(f"Total valid real images available: {len(valid_real_paths)}")

if len(valid_real_paths) < num_fakes:
    print("⚠️ Warning: More fakes than valid reals. We will use all valid reals.")
    num_to_sample = len(valid_real_paths)
else:
    num_to_sample = num_fakes

# דגימה רנדומלית מדויקת של תמונות אמיתיות תקינות כדי להשוות לכמות המזויפות
random.seed(42)
sampled_real_paths = random.sample(valid_real_paths, num_to_sample)
print(f"⚖️ Sampled {len(sampled_real_paths)} REAL images to match FAKE count.")

# --- שלב 4: העברת התמונות האמיתיות שנבחרו לתיקיית היעד ---
print("🚚 Copying selected Real images to final directory...")
missing_files = 0
for img_path in tqdm(sampled_real_paths, desc="Copying Reals"):
    if os.path.exists(img_path):
        shutil.copy(img_path, os.path.join(final_real_dir, os.path.basename(img_path)))
    else:
        missing_files += 1

if missing_files > 0:
    print(f"⚠️ Warning: {missing_files} files from JSON were not found on disk!")

# --- שלב 5: דחיסה ושמירה לדרייב בצורה מסודרת ---
print("\n🤐 Zipping the perfectly balanced dataset...")
# אנחנו נכנסים לתיקיית העבודה ודוחסים אותה כך שה-ZIP יכיל רק את התיקיות real ו-fake
!cd {final_dataset_dir} && zip -r -q /content/CelebA_HQ_Balanced_ZeroShot.zip real/ fake/

print("💾 Saving to Google Drive...")
!cp /content/CelebA_HQ_Balanced_ZeroShot.zip {final_zip_drive}

# בדיקת גדלים לסיום
if os.path.exists(final_zip_drive):
    drive_size = os.path.getsize(final_zip_drive) / (1024 * 1024)
    print(f"\n🎉 SUCCESS! The final balanced dataset is secured in your Drive.")
    print(f"📁 File Name: CelebA_HQ_Balanced_ZeroShot.zip")
    print(f"⚖️ Size: {drive_size:.2f} MB")
    print(f"📊 Contains: {len(os.listdir(final_fake_dir))} Fakes | {len(os.listdir(final_real_dir))} Reals")
else:
    print("\n🚨 Failed to copy to Drive! Do not close the session.")

🔌 Mounting Google Drive...
Mounted at /content/drive

📦 Extracting Fake images...
📊 Found 17643 FAKE images.
🚚 Moving Fake images to final directory...


Copying Fakes:   0%|          | 0/17643 [00:00<?, ?it/s]


📦 Extracting Real images...

📋 Loading valid JSON and balancing data...
Total valid real images available: 25195
⚖️ Sampled 17643 REAL images to match FAKE count.
🚚 Copying selected Real images to final directory...


Copying Reals:   0%|          | 0/17643 [00:00<?, ?it/s]


🤐 Zipping the perfectly balanced dataset...
💾 Saving to Google Drive...

🎉 SUCCESS! The final balanced dataset is secured in your Drive.
📁 File Name: CelebA_HQ_Balanced_ZeroShot.zip
⚖️ Size: 480.04 MB
📊 Contains: 17643 Fakes | 17643 Reals
